In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Chargement des données nettoyées 
df = pd.read_csv('../data/processed/data_cleaned.csv')
print(f"Dimensions du dataset : {df.shape}")

Dimensions du dataset : (4372, 58)


In [4]:
# CustomerID n'est jamais une feature prédictive
cols_to_exclude = [c for c in ["CustomerID", "Churn"] if c in df.columns]

X = df.drop(columns=cols_to_exclude)
y = df["Churn"]

# Split AVANT tout traitement pour éviter le data leakage
#random_state=42 : à chaque fois l'algorithme va mélanger et couper les données exactement de la même manière
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

Train : 3497 lignes | Test : 875 lignes


In [6]:
# On calcule le taux de valeurs manquantes (incluant 'Inconnu') uniquement sur le TRAIN
X_train_temp = X_train.replace("Inconnu", np.nan)
missing_rate = X_train_temp.isnull().mean()
cols_to_drop = missing_rate[missing_rate > 0.5].index.tolist()

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"Colonnes supprimées (>50% manquants) : {cols_to_drop}")

Colonnes supprimées (>50% manquants) : []


In [7]:
# SupportTicketsCount -> limité entre 0 et 20
if "SupportTicketsCount" in X_train.columns:
    X_train["SupportTicketsCount"] = X_train["SupportTicketsCount"].clip(lower=0, upper=20)
    X_test["SupportTicketsCount"] = X_test["SupportTicketsCount"].clip(lower=0, upper=20)

# SatisfactionScore -> les 0, -1 et 99 deviennent NaN, puis on borne entre 1 et 5
if "SatisfactionScore" in X_train.columns:
    for dataset in [X_train, X_test]:
        dataset["SatisfactionScore"] = dataset["SatisfactionScore"].replace(0, np.nan)
        dataset["SatisfactionScore"] = pd.to_numeric(dataset["SatisfactionScore"], errors="coerce").clip(lower=1, upper=5)

print("Valeurs aberrantes corrigées.")

Valeurs aberrantes corrigées.


In [8]:
# On liste les colonnes numériques ayant des NaN
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cols_with_nan = [c for c in num_cols if X_train[c].isna().any()]

# Mapping des stratégies d'imputation (médiane par défaut)
strategy_map = {"Age": "median", "SatisfactionScore": "median", "MonetaryTotal": "median"}

for col in cols_with_nan:
    strategy = strategy_map.get(col, "median")
    imputer = SimpleImputer(strategy=strategy)
    
    # Fit sur le train, transform sur le train et le test
    X_train[[col]] = imputer.fit_transform(X_train[[col]])
    X_test[[col]] = imputer.transform(X_test[[col]])

print(f"Colonnes imputées : {cols_with_nan}")

Colonnes imputées : ['AvgDaysBetweenPurchases', 'Age', 'SatisfactionScore']


In [9]:
# Mappings selon le script
mappings = {
    "LoyaltyLevel": {"Nouveau": 1, "Jeune": 2, "Établi": 3, "Ancien": 4},
    "AgeCategory": {"Inconnu": 0, "18-24": 1, "25-34": 2, "35-44": 3, "45-54": 4, "55-64": 5, "65+": 6},
    "SpendingCategory": {"Low": 1, "Medium": 2, "High": 3, "VIP": 4},
    "ChurnRiskCategory": {"Faible": 1, "Moyen": 2, "Élevé": 3, "Critique": 4},
    "BasketSizeCategory": {"Inconnu": 0, "Petit": 1, "Moyen": 2, "Grand": 3},
    "RFMSegment": {"Dormants": 0, "Potentiels": 1, "Fidèles": 2, "Champions": 3},
    "PreferredTimeOfDay": {"Nuit": 0, "Matin": 1, "Midi": 2, "Après-midi": 3, "Soir": 4}
}

for col, mapping in mappings.items():
    if col in X_train.columns:
        X_train[col] = X_train[col].map(mapping).fillna(0).astype(int)
        X_test[col]  = X_test[col].map(mapping).fillna(0).astype(int)

print("Encodage ordinal terminé.")

Encodage ordinal terminé.


In [10]:
ohe_cols = ["Gender", "WeekendPreference", "ProductDiversity", "FavoriteSeason", "Region", "AccountStatus", "CustomerType"]
cols = [c for c in ohe_cols if c in X_train.columns]

X_train = pd.get_dummies(X_train, columns=cols, drop_first=False)
X_test = pd.get_dummies(X_test, columns=cols, drop_first=False)

# Aligner le test sur le train pour s'assurer d'avoir les mêmes colonnes
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Conversion des booléens en entiers (0/1)
bool_cols = X_train.select_dtypes(include=["bool"]).columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

print("One-Hot Encoding terminé.")

One-Hot Encoding terminé.


In [11]:
if "Country" in X_train.columns:
    # 1. Regrouper les pays rares (<10 occurrences dans le TRAIN)
    country_counts = X_train["Country"].value_counts()
    rare_countries = country_counts[country_counts < 10].index.tolist()

    X_train["Country"] = X_train["Country"].where(~X_train["Country"].isin(rare_countries), "Other")
    X_test["Country"] = X_test["Country"].where(~X_test["Country"].isin(rare_countries), "Other")

    # 2. Calculer le taux de Churn moyen par pays sur le TRAIN
    churn_rate = pd.concat([X_train[["Country"]], y_train], axis=1).groupby("Country")["Churn"].mean()
    global_mean = y_train.mean()

    # 3. Appliquer le mapping
    X_train["Country"] = X_train["Country"].map(churn_rate).fillna(global_mean)
    X_test["Country"] = X_test["Country"].map(churn_rate).fillna(global_mean)

    print("Target Encoding appliqué sur 'Country'.")

Target Encoding appliqué sur 'Country'.


In [12]:
# Identifier les colonnes binaires (0/1) pour ne PAS les normaliser
def is_binary(series):
    unique_vals = set(series.dropna().unique())
    return unique_vals.issubset({0, 1, 0.0, 1.0})

binary_cols = [c for c in X_train.select_dtypes(include=[np.number]).columns if is_binary(X_train[c])]
num_cols = [c for c in X_train.select_dtypes(include=[np.number]).columns if c not in binary_cols]

scaler = StandardScaler()

# On ne normalise que les colonnes continues
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Colonnes continues normalisées : {len(num_cols)}")
print(f"Colonnes binaires laissées telles quelles : {len(binary_cols)}")

Colonnes continues normalisées : 48
Colonnes binaires laissées telles quelles : 33


In [14]:
# Vérification rapide
nan_train = X_train.isnull().sum().sum()
non_num = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"NaN résiduels : {nan_train} | Colonnes non numériques : {len(non_num)}")

# Sauvegarde
os.makedirs('../data/train_test', exist_ok=True)
X_train.to_csv('../data/train_test/X_train.csv', index=False)
X_test.to_csv('../data/train_test/X_test.csv', index=False)
y_train.to_csv('../data/train_test/y_train.csv', index=False)
y_test.to_csv('../data/train_test/y_test.csv', index=False)

print("Pipeline Notebook terminé. Fichiers sauvegardés pour la modélisation !")

NaN résiduels : 0 | Colonnes non numériques : 0
Pipeline Notebook terminé. Fichiers sauvegardés pour la modélisation !
